In [1]:
import pandas as pd 
import json
from pathlib import Path

In [2]:
# Current notebook location (or working directory)
current_dir = Path().resolve()

# Go up n directories (e.g., 2 up)
project_root = current_dir.parents[2]  # adjust number based on how far up your project root is

# Build path to CSV
csv_path = project_root / 'main' / 'Repository' / 'DataFlatFiles' / 'Tbl_Domestic_Fees_Vectors.csv'

# Load CSV
df_Data = pd.read_csv(csv_path)
df_Data.head(5)

,UniversityID,Ancillary Fees,Board only (on-campus meal plan),Books and Supplies,Other Expenses,Room and Board (on-campus),Room only (on-campus),Transportation (off-campus),Tuition Fees
0,2.0,1341.0,5288.0,900.0,1851.0,14118.0,8830.0,315.0,6089.0
1,3.0,1249.0,5420.0,1500.0,0.0,0.0,7875.0,0.0,6000.0
2,4.0,1286.0,2872.0,2200.0,1700.0,12946.0,8094.0,500.0,5985.0
3,5.0,1175.0,0.0,2000.0,18480.0,0.0,0.0,979.0,6052.0
4,6.0,1889.0,5300.0,1500.0,0.0,14831.0,9531.0,312.0,6118.0


In [3]:
df_Data['Target'] = df_Data['UniversityID']

In [4]:
print(df_Data.columns.tolist())

['UniversityID', 'Ancillary Fees', 'Board only (on-campus meal plan)', 'Books and Supplies', 'Other Expenses', 'Room and Board (on-campus)', 'Room only (on-campus)', 'Transportation (off-campus)', 'Tuition Fees', 'Target']


In [5]:
# Get feature columns (all except 'Target')
Feature_List = df_Data.drop(['Target','UniversityID'], axis=1).columns.tolist()

# Get target column(s)
Target = ['Target']  # simpler; or use: df_Data[['Target']].columns.tolist()

print(f"Features => {Feature_List}\nTarget => {Target}")


Features => ['Ancillary Fees', 'Board only (on-campus meal plan)', 'Books and Supplies', 'Other Expenses', 'Room and Board (on-campus)', 'Room only (on-campus)', 'Transportation (off-campus)', 'Tuition Fees']
Target => ['Target']


In [6]:
df_Data_Reduced = df_Data[Feature_List+Target]

Generating Domestic Fee Match Model

In [7]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [8]:
feetypes = df_Data_Reduced.drop(columns=["Target"])
universitids = df_Data_Reduced["Target"]

In [9]:
feetypes

,Ancillary Fees,Board only (on-campus meal plan),Books and Supplies,Other Expenses,Room and Board (on-campus),Room only (on-campus),Transportation (off-campus),Tuition Fees
0,1341.0,5288.0,900.0,1851.0,14118.0,8830.0,315.0,6089.0
1,1249.0,5420.0,1500.0,0.0,0.0,7875.0,0.0,6000.0
2,1286.0,2872.0,2200.0,1700.0,12946.0,8094.0,500.0,5985.0
3,1175.0,0.0,2000.0,18480.0,0.0,0.0,979.0,6052.0
4,1889.0,5300.0,1500.0,0.0,14831.0,9531.0,312.0,6118.0
5,1599.0,0.0,0.0,1416.0,0.0,7595.0,223.0,5781.0
6,1225.0,0.0,0.0,0.0,12305.0,9320.0,452.0,6067.0
7,1525.0,6326.0,1500.0,2350.0,14016.0,7690.0,314.0,6091.0
8,1492.0,5120.0,1600.0,0.0,11370.0,6250.0,242.0,6043.0
9,2111.0,6300.0,0.0,2403.0,14679.0,7775.0,300.0,5983.0


In [10]:
universitids

0      2.0
1      3.0
2      4.0
3      5.0
4      6.0
5      7.0
6      8.0
7      9.0
8     10.0
9     11.0
10    12.0
11    13.0
12    14.0
13    15.0
14    16.0
15    17.0
16    18.0
17    19.0
18    20.0
Name: Target, dtype: float64

In [11]:
scaler = StandardScaler()
feetypes_scaled = scaler.fit_transform(feetypes)

In [17]:
nn = NearestNeighbors(
    n_neighbors=6,
    metric='cosine'   
)

nn.fit(feetypes_scaled)

NearestNeighbors(metric='cosine', n_neighbors=6)

In [18]:
def recommend(user_input):

    user_scaled = scaler.transform([user_input])

    distances, indices = nn.kneighbors(user_scaled)

    return universitids.iloc[indices[0]].tolist()

In [19]:
sample_vector = df_Data[df_Data['UniversityID']==2.0].drop(['UniversityID','Target'],axis=1).values.tolist()[0]
sample_vector

[1341.0, 5288.0, 900.0, 1851.0, 14118.0, 8830.0, 315.0, 6089.0]

In [20]:
recommended_list = recommend(sample_vector)

df_Data[df_Data['UniversityID'].isin(recommended_list)]

c:\ProgramData\anaconda3\envs\myConda\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,UniversityID,Ancillary Fees,Board only (on-campus meal plan),Books and Supplies,Other Expenses,Room and Board (on-campus),Room only (on-campus),Transportation (off-campus),Tuition Fees,Target
0,2.0,1341.0,5288.0,900.0,1851.0,14118.0,8830.0,315.0,6089.0,2.0
4,6.0,1889.0,5300.0,1500.0,0.0,14831.0,9531.0,312.0,6118.0,6.0
7,9.0,1525.0,6326.0,1500.0,2350.0,14016.0,7690.0,314.0,6091.0,9.0
9,11.0,2111.0,6300.0,0.0,2403.0,14679.0,7775.0,300.0,5983.0,11.0
16,18.0,1557.0,6650.0,1178.0,1649.0,17086.0,9950.0,288.0,6050.0,18.0
17,19.0,1329.0,6548.0,1500.0,3200.0,15292.0,8744.0,700.0,6059.0,19.0


Saving Pkg 

In [21]:
import joblib

In [22]:
joblib.dump({
    "model": nn,
    "scaler": scaler,
    "universityids": universitids,
    "X_scaled": feetypes_scaled
}, "domestic_fees_matcher.pkl")

['domestic_fees_matcher.pkl']